# 06. 대화 요약 (Conversation Summary)

> 메시지를 그냥 삭제하면 맥락도 함께 사라져요. **증분 요약**을 노드로 끼워 넣어 토큰 비용은 줄이고 맥락은 유지하는 패턴을 구현해요.

## 학습 목표

이 노트북을 마치면 다음을 할 수 있어요:

1. 긴 대화에서 발생하는 컨텍스트 윈도우 문제를 이해하고 설명할 수 있어요
2. `MessagesState`를 확장하여 `summary` 필드를 가진 커스텀 상태를 정의할 수 있어요
3. 대화 길이를 기준으로 요약 노드로 라우팅하는 조건 함수를 작성할 수 있어요
4. `RemoveMessage`로 오래된 메시지를 삭제하면서 증분 요약(incremental summary)을 구현할 수 있어요

## 사전 지식

- `05-DeleteMessages.ipynb` — RemoveMessage, 히스토리 트리밍
- `04-Durable-Execution.ipynb` — MemorySaver, 체크포인터
- `StateGraph`의 노드·엣지·조건부 분기 기초


## 왜 대화 요약이 필요한가요?

대화를 지속하면 메시지가 계속 쌓입니다. 하지만 LLM은 처리할 수 있는 토큰 수가 한정되어 있어요.

앞서 `05-DeleteMessages`에서 오래된 메시지를 **단순 삭제**하는 방법을 배웠어요. 하지만 삭제하면 맥락을 잃어버려요. 마치 회의 녹음을 전부 지워버리는 것과 같죠. **대화 요약**은 녹음을 지우되, 핵심 내용을 회의록으로 남기는 것과 같아요.

| 방법 | 장점 | 단점 |
|------|------|------|
| 단순 삭제 (05번 노트북) | 구현 간단, 즉시 효과 | 맥락 유실, 이전 정보 참조 불가 |
| **대화 요약** (이번 노트북) | 맥락 보존, 토큰 절약 | 요약 비용 발생, 세부 정보 손실 가능 |

| 문제 | 영향 |
|------|------|
| 컨텍스트 윈도우 초과 | 오류 발생, 오래된 메시지 잘림 |
| 토큰 비용 증가 | 긴 대화일수록 비용이 급격히 증가 |
| 응답 지연 | 처리할 토큰이 많을수록 응답이 느려짐 |

**해결책**: 대화 기록을 주기적으로 요약하고, 오래된 메시지는 삭제해요. 요약본은 시스템 메시지로 주입하여 맥락을 유지해요.

> 🔑 **핵심 개념**: 대화 요약 패턴은 "모든 메시지를 보존" 대신 "맥락을 유지하면서 오래된 메시지를 압축"하는 전략이에요. 마치 회의록을 요약해 두고 원본 속기를 버리는 것과 같아요.

### 전체 아키텍처

```mermaid
flowchart TD
    A([START]) --> B[conversation<br/>대화 노드]
    B --> C{should_continue<br/>메시지 6개 초과?}
    C -- "아니오 (6개 이하)" --> D([END])
    C -- "예 (6개 초과)" --> E[summarize_conversation<br/>요약 노드]
    E --> D

    style A fill:#d4edda,stroke:#28a745,color:#155724
    style D fill:#d4edda,stroke:#28a745,color:#155724
    style B fill:#cce5ff,stroke:#007bff,color:#004085
    style C fill:#fff3cd,stroke:#ffc107,color:#856404
    style E fill:#e2d5f1,stroke:#6f42c1,color:#3d1f6e
```

### 상태 구조

| 필드 | 타입 | 설명 |
|------|------|------|
| `messages` | `list` | 대화 메시지 목록 (add_messages reducer) |
| `summary` | `str` | 이전 대화의 요약본 (초기값: 빈 문자열) |


## 1. 환경 설정


In [ ]:
# ---------------------------------------------------
# 환경 변수 로드
# ---------------------------------------------------
# OPENAI_API_KEY 등 API 키를 .env 파일에서 불러와요
from dotenv import load_dotenv

load_dotenv()


In [ ]:
# ---------------------------------------------------
# LangSmith 추적 설정 (선택)
# ---------------------------------------------------
# 실행 흐름을 LangSmith에서 시각적으로 확인할 수 있어요
import os

# os.environ["LANGCHAIN_TRACING_V2"] = "true"
# os.environ["LANGCHAIN_PROJECT"] = "LangGraph-04-ConversationSummary"


## 2. 커스텀 State 정의

`MessagesState`를 상속받아 `summary` 필드를 추가해요.
기본 `MessagesState`에는 `messages`만 있지만, 우리는 요약본을 별도로 저장할 공간이 필요해요.


In [ ]:
# ---------------------------------------------------
# 의존성 import
# ---------------------------------------------------
# Annotated: 타입 힌트에 메타데이터(reducer)를 붙여요
# Literal: 반환 타입을 특정 문자열 값으로 제한해요
from typing import Literal, Annotated

# init_chat_model: V1 방식의 모델 초기화 함수예요
from langchain.chat_models import init_chat_model

# 메시지 타입들 - 역할에 따라 다른 클래스를 사용해요
from langchain.messages import SystemMessage, RemoveMessage, HumanMessage

# MemorySaver: 인메모리 체크포인터 (프로덕션에서는 PostgresSaver 사용)
from langgraph.checkpoint.memory import MemorySaver

# MessagesState: messages 필드가 기본 포함된 상태 베이스 클래스
# StateGraph, START, END: 그래프 구성 핵심 요소
from langgraph.graph import MessagesState, StateGraph, START, END

# add_messages: 메시지를 덮어쓰지 않고 누적하는 reducer
from langgraph.graph.message import add_messages


In [ ]:
llm = init_chat_model("openai:gpt-4o-mini", temperature=0)

# TODO: 이 셀의 핵심 구현 코드를 수업 시간에 직접 작성하세요.
# 목표: 커스텀 State 클래스 정의
# 위 마크다운 설명과 힌트를 참고해 필요한 타입, 함수, 그래프 구성, 실행 코드를 완성하세요.


## 3. 노드 함수 정의

그래프에는 두 개의 노드가 있어요:
1. **`conversation`** — 사용자 메시지에 응답하는 일반 대화 노드
2. **`summarize_conversation`** — 메시지를 요약하고 오래된 메시지를 삭제하는 노드

### 3-1. conversation 노드 (ask_llm)


In [ ]:
# TODO: 이 셀의 핵심 구현 코드를 수업 시간에 직접 작성하세요.
# 목표: conversation 노드: LLM 호출
# 위 마크다운 설명과 힌트를 참고해 필요한 타입, 함수, 그래프 구성, 실행 코드를 완성하세요.


### 3-2. should_continue 조건 함수

대화 흐름을 결정하는 라우터 함수예요. 메시지 수를 검사하여 요약이 필요한지 판단해요.


In [ ]:
# TODO: 이 셀의 핵심 구현 코드를 수업 시간에 직접 작성하세요.
# 목표: 조건 함수: 요약 필요 여부 판단
# 위 마크다운 설명과 힌트를 참고해 필요한 타입, 함수, 그래프 구성, 실행 코드를 완성하세요.


### 3-3. summarize_conversation 노드

이 노드는 두 가지 역할을 동시에 수행해요:
1. **증분 요약 생성** -- 기존 요약이 있으면 확장하고, 없으면 새로 생성
2. **메시지 정리** -- 마지막 2개를 제외한 모든 메시지를 `RemoveMessage`로 삭제

### 신규 요약 vs 증분 요약

```mermaid
flowchart TD
    A{이전 요약이 있나요?} -->|없음| B["전체 대화를 요약<br/>(신규 생성)"]
    A -->|있음| C["기존 요약 + 새 메시지<br/>(증분 확장)"]
    B --> D["마지막 2개 메시지 제외<br/>나머지 RemoveMessage 삭제"]
    C --> D
    D --> E["summary 필드 업데이트"]
    
    classDef decision fill:#fff3cd,stroke:#ffc107,color:#856404
    classDef process fill:#cce5ff,stroke:#007bff,color:#004085
    classDef output fill:#d4edda,stroke:#28a745,color:#155724
    
    class A decision
    class B,C,D process
    class E output
```


In [ ]:
# TODO: 이 셀의 핵심 구현 코드를 수업 시간에 직접 작성하세요.
# 목표: summarize_conversation 노드: 요약 + 메시지 정리
# 위 마크다운 설명과 힌트를 참고해 필요한 타입, 함수, 그래프 구성, 실행 코드를 완성하세요.


## 4. 그래프 조립 및 컴파일


In [ ]:
# TODO: 이 셀의 핵심 구현 코드를 수업 시간에 직접 작성하세요.
# 목표: StateGraph 생성 및 노드 등록
# 위 마크다운 설명과 힌트를 참고해 필요한 타입, 함수, 그래프 구성, 실행 코드를 완성하세요.


In [ ]:
from IPython.display import Image, display

# TODO: 이 셀의 핵심 구현 코드를 수업 시간에 직접 작성하세요.
# 목표: 그래프 흐름: START → conversation → {should_continue} → summarize_conversation 또는 END
# 위 마크다운 설명과 힌트를 참고해 필요한 타입, 함수, 그래프 구성, 실행 코드를 완성하세요.


## 5. 그래프 실행 테스트

여러 메시지를 순서대로 보내면서 요약이 언제 트리거되는지 관찰해볼게요.

### 5-1. 스트리밍 헬퍼 함수 정의


In [ ]:
# TODO: 이 셀의 핵심 구현 코드를 수업 시간에 직접 작성하세요.
# 목표: 그래프 실행 결과를 스트리밍으로 출력하는 헬퍼 함수
# 위 마크다운 설명과 힌트를 참고해 필요한 타입, 함수, 그래프 구성, 실행 코드를 완성하세요.


### 5-2. 초기 대화 (요약 미발생 구간)


In [ ]:
# TODO: 이 셀의 핵심 구현 코드를 수업 시간에 직접 작성하세요.
# 목표: 스레드 설정
# 위 마크다운 설명과 힌트를 참고해 필요한 타입, 함수, 그래프 구성, 실행 코드를 완성하세요.


In [ ]:
# TODO: 이 셀의 핵심 구현 코드를 수업 시간에 직접 작성하세요.
# 목표: 두 번째 메시지: 이름 확인
# 위 마크다운 설명과 힌트를 참고해 필요한 타입, 함수, 그래프 구성, 실행 코드를 완성하세요.


In [ ]:
# TODO: 이 셀의 핵심 구현 코드를 수업 시간에 직접 작성하세요.
# 목표: 세 번째 메시지: 추가 정보
# 위 마크다운 설명과 힌트를 참고해 필요한 타입, 함수, 그래프 구성, 실행 코드를 완성하세요.


### 5-3. 요약 전 상태 확인

3턴(6개 메시지)이 쌓인 현재 상태를 확인해봐요. 아직 요약은 발생하지 않았어요.


In [ ]:
# TODO: 이 셀의 핵심 구현 코드를 수업 시간에 직접 작성하세요.
# 목표: 현재 그래프 상태 조회
# 위 마크다운 설명과 힌트를 참고해 필요한 타입, 함수, 그래프 구성, 실행 코드를 완성하세요.


### 5-4. 요약 트리거 (7번째 메시지)

4번째 메시지를 보내면 메시지가 7개가 되어 `should_continue`가 `summarize_conversation`을 선택해요.


In [ ]:
# TODO: 이 셀의 핵심 구현 코드를 수업 시간에 직접 작성하세요.
# 목표: 네 번째 메시지: 요약 트리거
# 위 마크다운 설명과 힌트를 참고해 필요한 타입, 함수, 그래프 구성, 실행 코드를 완성하세요.


### 5-5. 요약 후 상태 확인

요약이 완료된 후 상태를 확인해봐요. `summary` 필드에 요약본이 저장되고, `messages`에는 마지막 2개 메시지만 남아있어요.


In [ ]:
# TODO: 이 셀의 핵심 구현 코드를 수업 시간에 직접 작성하세요.
# 목표: 요약 후 상태 조회
# 위 마크다운 설명과 힌트를 참고해 필요한 타입, 함수, 그래프 구성, 실행 코드를 완성하세요.


### 5-6. 요약 기반 맥락 유지 확인

메시지 목록에는 2개밖에 없지만, 요약이 `SystemMessage`로 주입되어 이전 대화 내용을 기억하는지 확인해볼게요.


In [ ]:
# TODO: 이 셀의 핵심 구현 코드를 수업 시간에 직접 작성하세요.
# 목표: 요약 기반 장기 기억 확인
# 위 마크다운 설명과 힌트를 참고해 필요한 타입, 함수, 그래프 구성, 실행 코드를 완성하세요.


In [ ]:
# TODO: 이 셀의 핵심 구현 코드를 수업 시간에 직접 작성하세요.
# 목표: 직업 정보 기억 테스트
# 위 마크다운 설명과 힌트를 참고해 필요한 타입, 함수, 그래프 구성, 실행 코드를 완성하세요.


## 6. 증분 요약 동작 확인

대화가 더 길어지면 두 번째 요약이 생성될 때 기존 요약을 확장하는 증분 방식이 사용돼요.
새 대화를 시작하여 확인해봐요.


In [ ]:
# TODO: 이 셀의 핵심 구현 코드를 수업 시간에 직접 작성하세요.
# 목표: 증분 요약 테스트용 새 스레드
# 위 마크다운 설명과 힌트를 참고해 필요한 타입, 함수, 그래프 구성, 실행 코드를 완성하세요.


In [ ]:
# TODO: 이 셀의 핵심 구현 코드를 수업 시간에 직접 작성하세요.
# 목표: 1차 요약 결과 확인
# 위 마크다운 설명과 힌트를 참고해 필요한 타입, 함수, 그래프 구성, 실행 코드를 완성하세요.


In [ ]:
# TODO: 이 셀의 핵심 구현 코드를 수업 시간에 직접 작성하세요.
# 목표: 추가 대화로 2차 요약 트리거
# 위 마크다운 설명과 힌트를 참고해 필요한 타입, 함수, 그래프 구성, 실행 코드를 완성하세요.


In [ ]:
# TODO: 이 셀의 핵심 구현 코드를 수업 시간에 직접 작성하세요.
# 목표: 2차 요약 결과 확인 (증분 요약 검증)
# 위 마크다운 설명과 힌트를 참고해 필요한 타입, 함수, 그래프 구성, 실행 코드를 완성하세요.


## 7. 실습: 요약 임계값 조절

아래 코드를 수정하여 요약 임계값을 바꿔보세요.


In [ ]:
# ============================================================
# TODO: 요약 임계값을 4개로 낮춰서 더 자주 요약이 발생하도록 해보세요
# 힌트: should_continue 함수에서 len(messages) > 6을 변경해요
# 예상 결과: 2턴(4개 메시지) 이후부터 요약이 트리거돼요
# 추가 도전: 토큰 수 기준으로 바꿔보세요 (tiktoken 사용)
# ============================================================


## 핵심 요약

이 노트북에서 다음 내용을 배웠어요:

- **컨텍스트 윈도우 문제**: 긴 대화는 토큰 비용 증가·지연·오류를 유발하며, 요약 패턴으로 해결할 수 있어요
- **커스텀 State**: `MessagesState`를 상속하고 `summary: str` 필드를 추가하여 요약본 저장 공간을 마련해요
- **SystemMessage 주입**: `summary`가 있을 때 `SystemMessage`로 LLM에 전달하여 삭제된 메시지의 맥락을 보존해요
- **조건부 라우팅**: `should_continue`에서 메시지 수를 검사하여 요약 노드 진입을 제어해요
- **증분 요약**: 기존 요약이 있으면 새 내용을 덧붙이는 방식으로 요약 비용을 절약해요
- **RemoveMessage**: 마지막 N개를 제외한 오래된 메시지를 삭제하여 상태를 경량화해요

## 다음 노트북 예고

다음 `07-Streaming-Steps.ipynb`에서는 **스텝 단위 스트리밍**을 배워요. 그래프의 각 노드 실행 결과를 실시간으로 추적하고, 중간 결과를 사용자에게 스트리밍하는 방법을 살펴볼게요.
